In [8]:
# Importing required libraries
import pandas as pd
import os
import pyodbc
import urllib
from sqlalchemy import text
from sqlalchemy import create_engine

In [3]:
# SQL Server connection configuration
connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=SAM\\SQLEXPRESS;"
    "DATABASE=gig_welfare_india;"
    "Trusted_Connection=yes;")

In [5]:
# Encode connection string and create SQLAlchemy engine
params = urllib.parse.quote_plus(connection_string)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

with engine.connect() as conn:
    print("Connected")

Connected


In [6]:
# Path containing all cleaned CSV datasets
CLEANED_PATH = r'C:\Users\Galbo\OneDrive\문서\Cleaned GIG'

In [7]:
# Dictionary mapping SQL table names to cleaned CSV files
tables = {"gig_workers_gender"      : "clean_gig_workers_gender.csv",
    "total_unorganised_27mar" : "clean_total_unorganised_27mar.csv",
    "total_unorganised_23mar" : "clean_total_unorganised_23mar.csv",
    "plfs_unemployment"       : "clean_plfs_unemployment.csv",
    "state_population"        : "clean_state_population.csv",
    "policy_timeline"         : "clean_policy_timeline.csv",
    "policy_lag"              : "clean_policy_lag.csv",
    "state_coverage"          : "clean_state_coverage.csv",
    "state_nsdp"              : "clean_state_nsdp.csv",
    "master_analytics_table"  : "master_analytics_table.csv",}

# Loopin through each dataset and upload it to SQL Server
for table_name, file_name in tables.items():
    df = pd.read_csv(os.path.join(CLEANED_PATH, file_name))
    df.to_sql(
        name      = table_name,
        con       = engine,
        if_exists = "replace",
        index     = False,
        schema    = "dbo"
    )
    print(f"{table_name:35} → {len(df):>3} rows uploaded")

 gig_workers_gender                  →  36 rows uploaded
 total_unorganised_27mar             →  36 rows uploaded
 total_unorganised_23mar             →  36 rows uploaded
 plfs_unemployment                   → 185 rows uploaded
 state_population                    →  36 rows uploaded
 policy_timeline                     →  25 rows uploaded
 policy_lag                          →   5 rows uploaded
 state_coverage                      →  36 rows uploaded
 state_nsdp                          →  36 rows uploaded
 master_analytics_table              →  36 rows uploaded


In [9]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT TABLE_NAME, 
               (SELECT COUNT(*) 
                FROM INFORMATION_SCHEMA.COLUMNS c 
                WHERE c.TABLE_NAME = t.TABLE_NAME) as col_count
        FROM INFORMATION_SCHEMA.TABLES t
        WHERE TABLE_TYPE = 'BASE TABLE'
        ORDER BY TABLE_NAME """))
    df_tables = pd.DataFrame(result.fetchall(), columns=['TABLE_NAME', 'col_count'])

df_tables


,TABLE_NAME,col_count
0,gig_workers_gender,7
1,master_analytics_table,30
2,plfs_unemployment,5
3,policy_lag,4
4,policy_timeline,7
5,state_coverage,10
6,state_nsdp,13
7,state_population,3
8,total_unorganised_23mar,3
9,total_unorganised_27mar,3
